![Python Banner](../../Image/05_06_Baysian_CausalML.png)



# 6.3 Bayesian Structural Causal Models



A structural equation writes the outcome as a function of its direct causes and a disturbance. When the treatment disturbance and the outcome disturbance are correlated, ordinary least squares attributes part of that shared disturbance to the treatment. This notebook is the detail chapter for the structural family in `pycausalml.bayesian_causalML`:

| Class or function | Role |
|---|---|
| `simulate_structural_scm` | A confounded linear system with known effect $\alpha$, two instruments, and one outcome-only covariate |
| `BayesianStructuralSCM` | The cross-sectional model. Exclusion restrictions select two-stage least squares; otherwise the fit is an outcome regression |
| `BayesianTimeSeriesSCM` | A lagged structural system. Contemporaneous parents form a DAG. Interventions are rolled forward |

The cross-sectional specification follows the CausalPy note on [Bayesian structural causal models](https://causalpy.readthedocs.io/en/latest/knowledgebase/structural_causal_models.html). The coefficient posterior is a Bayesian bootstrap of both stages, or a ridge-normal posterior when no instrument is declared.



## Overview

### Why a structural equation, rather than a design dummy?

Difference-in-differences, a cutoff, and a single interrupted series each encode the intervention as a column in one regression. A structural model writes a separate assignment for the treatment and for the outcome, and it states which covariates are allowed in each assignment. That exclusion list is the identifying assumption.

The same idea extends one step in time. A protein, a vital sign, or a price can cause another series within the period and through its own lag. `BayesianTimeSeriesSCM` gives each series a Gaussian posterior and replaces an equation when the intervention is a hard `do`.

### What this tutorial covers

- The two-equation system, the role of $\rho = \mathrm{Corr}(U, V)$, and the OLS bias it produces
- How `instruments` and `outcome_only` partition the columns
- Recovery of a known $\alpha$ with `simulate_structural_scm`, and the failure of the regression that uses every column
- Potential outcomes $Y(0)$ and $Y(1)$ from a homogeneous structural effect
- A lagged system with a known contemporaneous coefficient, an impulse response, and a hard intervention
- Card's college-proximity design on the CausalPy `schoolReturns` file
- One patient's hourly lactate and sepsis-risk series from `Data/labs_timeseries.csv`



## The algorithms

### 1. The cross-sectional structural model

Index units by $i$ and suppress the index. The assignments are

$$
\begin{aligned}
T &= a_T + X_T^\top \beta_T + V, \\
Y &= \alpha T + a_Y + X_Y^\top \beta_Y + U,
\end{aligned}
$$

with $\mathrm{Corr}(U, V) = \rho$. The parameter of interest is $\alpha$, the structural coefficient on an endogenous treatment. Substituting the treatment equation into the outcome equation shows the OLS problem. The composite disturbance in the outcome still contains $V$ whenever $\rho \neq 0$, and $V$ is inside $T$. A regression of $Y$ on $T$ and the observed covariates therefore converges to $\alpha$ plus a term that depends on $\rho$ and on the variance of $V$.

`simulate_structural_scm` builds this system with five standardized covariates:

| Columns | Role in the simulation | Where they enter |
|---|---|---|
| 0 and 1 | Confounders | Treatment equation and outcome equation |
| 2 and 3 | Instruments | Treatment equation only |
| 4 | Outcome-only covariate | Outcome equation only |

The treatment and outcome disturbances are bivariate normal with correlation `rho`. The structural coefficient is the argument `alpha`. Set `binary_treatment=True` to threshold the latent treatment index at zero.

### 2. Exclusion and the estimator

`BayesianStructuralSCM.fit` partitions the columns of $X$:

- `instruments` enter the treatment equation only
- `outcome_only` enters the outcome equation only
- every remaining column is a confounder and enters both

A column cannot be both an instrument and outcome-only. If `instruments` is omitted, the treatment is taken to be conditionally ignorable given $X$ and the fit is a ridge-normal outcome regression. `identified_by_` records which of the two routes was used.

When instruments are present the point estimator is two-stage least squares.

1. Regress $T$ on an intercept, the confounders, and the instruments.
2. Regress $Y$ on an intercept, the fitted treatment, the confounders, and the outcome-only columns.

Posterior draws repeat both stages with Dirichlet weights on the rows (a Bayesian bootstrap). The weight vector treats each unit as exchangeable. First-stage uncertainty is inside every draw of $\alpha$. There is no separate delta-method correction.

Predictions under $do(T=t)$ use the outcome equation only. The first-stage residual is an estimation device. It is not part of the intervened assignment. With a homogeneous $\alpha$,

$$
Y(1) - Y(0) = \alpha
$$

for every covariate row, so the average of $Y(1) - Y(0)$ equals the structural coefficient. `potential_outcomes` and `predict_interval` implement that calculation. The interval is a pointwise credible band from the coefficient draws.

### 3. The lagged structural model

Let $X_t \in \mathbb{R}^d$ be the vector of series at time $t$. Contemporaneous parents form a DAG, stored again as `CausalGraph`. For each series $i$, with lag $L \ge 1$,

$$
\begin{aligned}
X_{i,t} = a_i
 &+ \sum_{j \in \mathrm{pa}(i)} \beta_{ij} X_{j,t}
 + \sum_{\ell=1}^{L} \phi_{i\ell} X_{i, t-\ell}
 + \sum_{\ell=1}^{L} \sum_{j \in \mathrm{pa}(i)} \psi_{ij\ell} X_{j, t-\ell}
 + \varepsilon_{i,t}.
\end{aligned}
$$

Each equation has its own ridge-normal posterior. An empty edge list is a system of independent autoregressions: contemporaneous parents are absent, and each series still depends on its own lags.

A **hard intervention** replaces the equation for the named series over the forecast horizon. `counterfactual` rolls the posterior-mean coefficients forward under the intervention and under a baseline. The baseline can be the observational forecast (`baseline=None`) or a second intervention, for example $do(X=1)$ against $do(X=0)$. Draws of the coefficients produce a 95% band on the effect path $X^{\mathrm{cf}}_t - X^{\mathrm{base}}_t$.

An **impulse** is a different operation. `impulse_response` adds a one-period shock to the structural residual and then lets every equation return to its fitted form. The shocked series is free to move on the next date. A hard `do` is not.

Both rollouts are conditional on the last `lag` observations of the fitted sample, unless `history` supplies another window.



## Setup

The estimators come from **[PyCausalML](https://github.com/zia207/PyCausalML)** (`pycausalml.bayesian_causalML`). The schooling file is the CausalPy `schoolReturns` table, from the [CausalPy data directory](https://github.com/pymc-labs/CausalPy/tree/main/causalpy/data), Apache License 2.0. The hourly series is `Data/labs_timeseries.csv`.


In [ ]:
import subprocess
import sys
from pathlib import Path

def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "pyproject.toml").is_file() and (cand / "src" / "pycausalml").is_dir():
            return cand
    raise FileNotFoundError(
        "PyCausalML repository was not found. Run this notebook from the repository."
    )

root = _repo_root()
data_dir = root / "Data"
if not data_dir.is_dir():
    raise FileNotFoundError(
        "The Data directory was not found. It should sit next to pyproject.toml."
    )

try:
    import pycausalml  # noqa: F401
    print("PyCausalML ready.")
except ImportError:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{root}[plot]"],
    )
    print("PyCausalML installed.")
print("Data directory:", data_dir)


In [ ]:

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pycausalml.bayesian_causalML import (
    BayesianStructuralSCM,
    BayesianTimeSeriesSCM,
    simulate_structural_scm,
)

plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
CAUSALPY_FILES = {
    "banks": "banks.csv",
    "drinking": "drinking.csv",
    "risk": "AJR2001.csv",
    "schoolReturns": "schoolingReturns.csv",
    "california_prop99": "california_prop99.csv",
}

def load_causalpy(name: str) -> pd.DataFrame:
    """Load a CausalPy example dataset.

    Uses causalpy.load_data when that package is installed. Otherwise reads
    the same CSV from the CausalPy GitHub data directory.
    """
    try:
        import causalpy as cp
        frame = cp.load_data(name)
        print(f"Loaded {name!r} with causalpy.load_data. Shape {frame.shape}.")
        return frame
    except ImportError:
        url = (
            "https://raw.githubusercontent.com/pymc-labs/CausalPy/main/causalpy/data/"
            + CAUSALPY_FILES[name]
        )
        frame = pd.read_csv(url)
        print(f"CausalPy is not installed. Loaded {name!r} from")
        print(url)
        print("Install the package with: pip install 'pycausalml[causalpy]'")
        print(f"Shape {frame.shape}.")
        return frame


def show_posterior(draws, estimate, title, truth=None):
    draws = np.asarray(draws, dtype=float).ravel()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(draws, bins=30, color="#4C72B0", alpha=0.9, density=True)
    ax.axvline(estimate, color="#8B1E3F", lw=2, label=f"estimate {estimate:.3f}")
    low, high = np.quantile(draws, [0.025, 0.975])
    ax.axvspan(low, high, color="#4C72B0", alpha=0.18, label="95% credible interval")
    if truth is not None:
        ax.axvline(truth, color="black", ls="--", lw=1.5, label=f"known effect {truth:.3f}")
    ax.set_xlabel("Structural coefficient")
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()



## Getting started: a known structural coefficient

`simulate_structural_scm` draws $n = 2500$ units with $\alpha = 3$ and $\rho = 0.6$. The comparison regression puts the treatment and all five covariates in one equation for $Y$. That regression is inconsistent for $\alpha$ because the instruments do not belong in the outcome equation and the shared disturbance remains inside $T$.


In [ ]:

sim = simulate_structural_scm(n=2500, alpha=3.0, rho=0.6, random_state=0)
design = np.column_stack([np.ones(len(sim["y"])), sim["treatment"], sim["X"]])
ols_slope = np.linalg.lstsq(design, sim["y"], rcond=None)[0][1]

structural = BayesianStructuralSCM(n_draws=80, random_state=0)
structural.fit(
    sim["X"],
    sim["treatment"],
    sim["y"],
    instruments=sim["instruments"],
    outcome_only=sim["outcome_only"],
    feature_names=sim["feature_names"],
)
print("Identified by:", structural.identified_by_)
print("OLS slope on treatment, all covariates included:", round(float(ols_slope), 3))
print(pd.Series(structural.summary()).drop(labels=["identified_by"]).astype(float).round(3))
print("identified_by:", structural.summary()["identified_by"])

y0, y1 = structural.potential_outcomes(sim["X"])
print("Mean Y(1) - Y(0):", round(float(np.mean(y1 - y0)), 4))
print("Structural estimate:  ", round(structural.effect_, 4))
show_posterior(structural.draws_, structural.effect_, "Cross-sectional structural model", truth=3.0)

grid = np.linspace(sim["treatment"].min(), sim["treatment"].max(), 40)
x_mean = np.repeat(sim["X"].mean(axis=0, keepdims=True), len(grid), axis=0)
mean, low, high = structural.predict_interval(x_mean, grid)
fig, ax = plt.subplots(figsize=(7, 4))
ax.fill_between(grid, low, high, color="#4C72B0", alpha=0.2, label="95% band at mean covariates")
ax.plot(grid, mean, color="#8B1E3F", label="E[Y | do(T=t), mean X]")
ax.set_xlabel("Intervened treatment")
ax.set_ylabel("Outcome")
ax.set_title("Outcome equation under do(T)")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()



The bootstrap interval covers $3$. The OLS slope is farther from $3$ than the structural estimate. Because $\alpha$ is constant, the average of the two potential outcomes differs from the estimate only by numerical error. The band in the figure is the uncertainty in $\alpha$ and in the intercept at the average covariate row. It is not a prediction interval for a new unit's disturbance.



## Getting started: a lagged system with a known coefficient

The recursion, for $t \ge 1$, is

$$
X_t = 0.4\, X_{t-1} + \varepsilon^X_t, \qquad
Y_t = 0.8\, X_t + 0.2\, Y_{t-1} + \varepsilon^Y_t.
$$

The contemporaneous structural coefficient of $X$ in the equation for $Y$ is $0.8$. The edge list is `(x, y)`. Lag 1 also enters $Y_{t-1}$ and $X_{t-1}$.


In [ ]:

rng = np.random.default_rng(0)
n_time = 1500
x = np.zeros(n_time)
y = np.zeros(n_time)
for t in range(1, n_time):
    x[t] = 0.4 * x[t - 1] + rng.normal(0.0, 0.5)
    y[t] = 0.8 * x[t] + 0.2 * y[t - 1] + rng.normal(0.0, 0.3)
series = pd.DataFrame({"x": x, "y": y})

lagged = BayesianTimeSeriesSCM([("x", "y")], lag=1, n_draws=40, random_state=0)
lagged.fit(series)
print(pd.Series(lagged.summary("y")["coef"], index=lagged.summary("y")["feature_names"]).round(3))

impulse = lagged.impulse_response("x", shock=1.0, horizon=12)
held = lagged.counterfactual(8, interventions={"x": 1.0}, baseline={"x": 0.0})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(impulse["effect"][:, series.columns.get_loc("y")], marker="o", color="#4C72B0")
axes[0].axhline(0.8, color="black", ls="--", lw=1, label="contemporaneous coefficient 0.8")
axes[0].set_title("Impulse: one-period shock to x")
axes[0].set_xlabel("Horizon")
axes[0].set_ylabel("Effect on y")
axes[0].legend(frameon=False)

steps = np.arange(held["effect"].shape[0])
y_index = series.columns.get_loc("y")
axes[1].fill_between(
    steps, held["effect_lower"][:, y_index], held["effect_upper"][:, y_index],
    color="#4C72B0", alpha=0.2, label="95% band",
)
axes[1].plot(steps, held["effect"][:, y_index], color="#8B1E3F", marker="o", label="do(x=1) minus do(x=0)")
axes[1].set_title("Hard intervention held for 8 periods")
axes[1].set_xlabel("Horizon")
axes[1].set_ylabel("Effect on y")
axes[1].legend(frameon=False)
fig.tight_layout()
plt.show()



The posterior mean of `x[t]` in the equation for $Y$ sits next to $0.8$. The impulse jumps by about that amount on impact and then decays, because the shock lasts one period and $X$ is allowed to return to its autoregression. The hard intervention holds $X$ at 1 rather than at 0 for the whole horizon, so the path on $Y$ stays near the contemporaneous coefficient plus the contribution of $Y$'s own lag. The band is coefficient uncertainty along that path.



### Key parameters

| Parameter | Where | Role |
|---|---|---|
| `alpha`, `rho` | `simulate_structural_scm` | Known structural coefficient and disturbance correlation. |
| `instruments`, `outcome_only` | `BayesianStructuralSCM.fit` | Exclusion restrictions. Other columns are confounders. |
| `n_draws` | both classes | Bayesian-bootstrap draws, or coefficient draws in the time-series rollout. |
| `edges` | `BayesianTimeSeriesSCM` | Contemporaneous parents. An empty list leaves independent autoregressions. |
| `lag` | `BayesianTimeSeriesSCM` | Number of lags of the series and of its contemporaneous parents. |
| `interventions` | `counterfactual` | A scalar or a horizon-length sequence that replaces an equation. |



## Use case: returns to schooling

Card (1995) uses growing up near a four-year college as a source of variation in years of education. The outcome here is log wage. Experience enters both the education equation and the wage equation, so it is a confounder in the partition above. `nearcollege` enters the education equation only. That exclusion is the identifying assumption: proximity affects wages through schooling and experience, not through a separate channel. The file is CausalPy's `schoolReturns` table (`schoolingReturns.csv`).


### Load the schooling table


In [ ]:

school = load_causalpy("schoolReturns")
school = school.dropna(subset=["wage", "education", "nearcollege", "experience"]).copy()
school = school.loc[school["wage"] > 0].copy()
school["log_wage"] = np.log(school["wage"].to_numpy(float))
school["near"] = (school["nearcollege"].astype(str) == "yes").astype(float)
print("Rows:", len(school))
print(school[["log_wage", "education", "experience", "near"]].describe().T[["mean", "std"]].round(3))


### Data visualization


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for flag, label, color in [(1.0, "Near a college", "#8B1E3F"), (0.0, "Not near a college", "#4C72B0")]:
    subset = school.loc[school["near"] == flag, "education"]
    axes[0].hist(subset, bins=18, alpha=0.55, label=label, color=color, density=True)
axes[0].set_xlabel("Years of education")
axes[0].set_ylabel("Density")
axes[0].set_title("First stage: proximity and schooling")
axes[0].legend(frameon=False)
axes[1].scatter(school["education"], school["log_wage"], s=8, alpha=0.15, c="#4C72B0")
axes[1].set_xlabel("Years of education")
axes[1].set_ylabel("Log wage")
axes[1].set_title("Observational wage-schooling cloud")
fig.tight_layout()
plt.show()



### Fit the structural model

The design matrix carries `experience` and `near`. The instrument list contains only `near`. Experience is then a confounder by the partition rule.


In [ ]:

covariates = school[["experience", "near"]]
ols_design = np.column_stack([
    np.ones(len(school)),
    school["education"].to_numpy(float),
    school["experience"].to_numpy(float),
])
ols_education = np.linalg.lstsq(ols_design, school["log_wage"].to_numpy(float), rcond=None)[0][1]

card = BayesianStructuralSCM(n_draws=80, random_state=0)
card.fit(
    covariates,
    school["education"].to_numpy(float),
    school["log_wage"].to_numpy(float),
    instruments=["near"],
)
print("Identified by:", card.identified_by_)
print(f"OLS coefficient on education, controlling for experience: {ols_education:.3f}")
print(pd.Series({k: v for k, v in card.summary().items() if k != "identified_by"}).round(3))
show_posterior(card.draws_, card.effect_, "Return to one year of schooling")

y0, y1 = card.potential_outcomes(covariates)
print("Mean structural contrast Y(1) - Y(0):", round(float(np.mean(y1 - y0)), 3))



The estimate is a log-wage change for one additional year of schooling under the exclusion restriction. The OLS coefficient that controls for experience answers a different equation: it leaves the education residual correlated with the wage residual. The credible interval is the Bayesian bootstrap of both stages. A wide interval is the sample's statement about a binary instrument. It is not a failure of the optimizer.

The homogeneous-effect calculation $Y(1) - Y(0)$ equals the coefficient for every person. The file does not identify a different return for every level of experience unless that interaction is written into the outcome equation.



## Use case: lactate and sepsis risk, one hospital stay

`Data/labs_timeseries.csv` is an hourly laboratory panel. The series used here is the longest stay, patient 6: lactate and `sepsis_risk_score` for 72 hours. The contemporaneous edge is lactate into the risk score. Lactate is a clinical marker that moves with perfusion; the risk score is a derived label. The fit is a description of this stay under that one-edge skeleton. It is not an estimate of the effect of treating lactate in a population.


In [ ]:

labs = pd.read_csv(data_dir / "labs_timeseries.csv")
stay_id = int(labs.groupby("patient_id").size().idxmax())
stay = (
    labs.loc[labs["patient_id"] == stay_id, ["hour_from_admission", "lactate", "sepsis_risk_score"]]
    .sort_values("hour_from_admission")
    .reset_index(drop=True)
)
print(f"Patient {stay_id}, hours: {len(stay)}")
print(stay[["lactate", "sepsis_risk_score"]].describe().T[["mean", "std"]].round(3))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(stay["hour_from_admission"], stay["lactate"], color="#4C72B0", label="Lactate")
ax.set_ylabel("Lactate", color="#4C72B0")
ax2 = ax.twinx()
ax2.plot(stay["hour_from_admission"], stay["sepsis_risk_score"], color="#8B1E3F", label="Sepsis risk")
ax2.set_ylabel("Sepsis risk score", color="#8B1E3F")
ax.set_xlabel("Hour from admission")
ax.set_title(f"Patient {stay_id}")
fig.tight_layout()
plt.show()


In [ ]:

clinical = stay.rename(columns={"sepsis_risk_score": "risk"})[["lactate", "risk"]]
clinical_model = BayesianTimeSeriesSCM(
    [("lactate", "risk")], lag=1, n_draws=40, random_state=0
)
clinical_model.fit(clinical)
print("Risk equation:")
print(pd.Series(
    clinical_model.summary("risk")["coef"],
    index=clinical_model.summary("risk")["feature_names"],
).round(3))

shock = clinical_model.impulse_response("lactate", shock=1.0, horizon=10)
risk_col = clinical.columns.get_loc("risk")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(shock["effect"][:, risk_col], marker="o", color="#8B1E3F")
ax.axhline(0.0, color="black", lw=1)
ax.set_xlabel("Hours after a one-unit lactate shock")
ax.set_ylabel("Change in sepsis risk score")
ax.set_title("Impulse response along the fitted stay")
fig.tight_layout()
plt.show()



`lactate[t]` in the risk equation is the contemporaneous structural coefficient for this stay. The impulse adds one lactate unit to the residual at the first forecast hour and then lets both equations run. Hours after the shock move because of the lags, not because lactate is held at the shocked value. A treatment policy that sets lactate to a target for many hours is the hard `counterfactual` path, and it needs a baseline policy to subtract. With 72 hours and two series, that band is a description of one trajectory.



## Summary and conclusion

`BayesianStructuralSCM` separates the treatment assignment from the outcome assignment. Instruments identify $\alpha$ when the treatment disturbance and the outcome disturbance are correlated. The Bayesian bootstrap carries the first stage into the posterior of $\alpha$. Predictions under $do(T)$ drop the first-stage residual.

`BayesianTimeSeriesSCM` uses the same Gaussian equation posterior one lag at a time. A one-period residual shock and a multi-period hard intervention are different experiments, and the two plotting panels keep them apart.

On the simulated cross-section the posterior covers $\alpha = 3$ and the saturated OLS slope does not. On the simulated lag the coefficient on contemporaneous $X$ covers $0.8$. The schooling fit is the same estimator with Card's exclusion restriction. The lactate fit is the same lagged estimator on one hospital stay.



## Scientific terminology

| Term | Meaning in this notebook |
|---|---|
| Structural coefficient | The direct effect written in an assignment, here $\alpha$ on endogenous $T$. |
| Endogeneity | Correlation between a regressor and the disturbance of the equation it sits in. |
| Exclusion restriction | A covariate is allowed in one assignment and shut out of the other. |
| Confounder column | A column that enters both the treatment equation and the outcome equation. |
| Bayesian bootstrap | Dirichlet weights on the sample rows. Each weighted two-stage fit is one posterior draw. |
| $Y(1), Y(0)$ | Outcomes under $do(T=1)$ and $do(T=0)$. With constant $\alpha$ their difference is $\alpha$. |
| Hard intervention | The structural equation is replaced for the whole horizon. |
| Impulse response | A one-period shock to a structural residual. Later hours use the fitted equations. |
